# 🚀 Q&A System V2 - Clean Modular Interface

Refactored UI with extracted modules for better maintainability.

In [1]:
# Setup and imports
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

# Import core system setup
from qanda_module.config import setup_system
from qanda_module.ui_gradio import launch_ui_with_toggle

print("📦 Imports successful!")

📦 Imports successful!


In [2]:
# System setup
print("🚀 Setting up system...")

helpers, qa_chain, config = setup_system("../config.yaml")

print("✅ System setup complete!")
print(f"Model: {config.chat_model_name}")
print(f"Temperature: {config.temperature}")
print(f"Debug: {config.debug_enabled}")

🚀 Setting up system...
Loaded 475 panelist profile URLs
Total panellist responses: 17815
Panellist responses with missing speaker_id: 0
✅ System setup complete!
Model: gpt-4o-mini
Temperature: 0.3
Debug: False


In [3]:
# Add this debug cell to see the exact error:
try:
    from qanda_module.database_clean import load_database_clean
    print("✅ database_clean import successful")
except ImportError as e:
    print(f"❌ database_clean import failed: {e}")

try:
    from qanda_module.legacy_imports import setup_qa_chain, ImprovedQAHelpers
    print("✅ legacy_imports import successful")
except ImportError as e:
    print(f"❌ legacy_imports import failed: {e}")

try:
    from chromadb import PersistentClient
    print("✅ chromadb import successful")
except ImportError as e:
    print(f"❌ chromadb import failed: {e}")

✅ database_clean import successful
✅ legacy_imports import successful
✅ chromadb import successful


In [4]:
print(f"helpers: {helpers}")
print(f"qa_chain: {qa_chain}")  
print(f"config: {config}")

helpers: <qanda_module.legacy_imports.ImprovedQAHelpers object at 0x162b70d40>
qa_chain: verbose=False combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Please answer the question based on the context.\n\nContext: {context}\n\nQuestion: {question}'), llm=ChatOpenAI(callbacks=[], client=<openai.resources.chat.completions.completions.Completions object at 0x166f6c920>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x12626ca70>, root_client=<openai.OpenAI object at 0x162dac830>, root_async_client=<openai.AsyncOpenAI object at 0x125feb860>, model_name='gpt-4o-mini', temperature=0.3, model_kwargs={}, openai_api_key=SecretStr('**********'), streaming=True), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variabl

In [5]:
# UI Selection and Launch
USE_NEW_DESIGN = True  # Toggle between current and new UI

demo = launch_ui_with_toggle(
    helpers, 
    qa_chain, 
    config, 
    design="new" if USE_NEW_DESIGN else "current"
)

print(f"🎯 Demo created using {'NEW' if USE_NEW_DESIGN else 'CURRENT'} design")

🎨 Creating new UI...
Loaded 475 panelists using date-based latest episode logic
✅ UI data prepared: 100 panelists loaded
✅ Successfully loaded 25 real episodes for scroller
✅ New semantic UI created
🎯 Demo created using NEW design


In [13]:
print(config.chroma_path)
print(config.duck_db_name)

../data/chroma_db_hf_may
../data/qanda_may.duckdb


In [8]:
# Launch the interface
print("🌐 Launching Gradio interface...")

demo.launch(
    share=True,  # Set to True for public link
    server_port=7860,
    show_error=True,
    debug=True  # Shows more info in console
)

🌐 Launching Gradio interface...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b72b925f2000a2c208.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🔍 Processing question: How do panelists view immigration policy?
🔍 Parameters: k=80, style=Balanced
Processing: How do panelists view immigration policy?...
✅ Question processed successfully
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b72b925f2000a2c208.gradio.live


In [ ]:
# Optional: Quick testing and experiments

def quick_test():
    """Test the system with a simple query."""
    from qanda_module.ui_gradio import handle_question
    
    test_question = "What did panelists say about climate change?"
    result = handle_question(test_question, 10, "Concise", helpers, qa_chain, config)
    
    print(f"✅ Test completed: {result[2]}")
    print(f"Answer length: {len(result[0])} chars")
    return result

# Uncomment to run test:
# test_result = quick_test()

In [14]:
# Stop the demo when needed
demo.close()

Closing server running on port: 7860


In [6]:
# Create database connection
import duckdb

# Using your config (assuming you already have helpers, qa_chain, config loaded)
con = duckdb.connect(config.duck_db_name)

# Or directly with the path if you prefer:
# con = duckdb.connect("../data/qanda_v2.duckdb")

In [ ]:
# Check if we're getting the right "latest" entries
con.execute("""
    SELECT name, id, profession, link, 
           ROW_NUMBER() OVER (PARTITION BY name ORDER BY id DESC) as rn
    FROM dim_panellist 
    WHERE name IN ('Christopher Pyne', 'Malcolm Turnbull', 'Penny Wong')
    ORDER BY name, id DESC
""").df()

In [ ]:
# Check panelist entries
validation_result = con.execute("""
    SELECT name, id, profession, link, 
           ROW_NUMBER() OVER (PARTITION BY name ORDER BY id DESC) as rn
    FROM dim_panellist 
    WHERE name IN ('Christopher Pyne', 'Malcolm Turnbull', 'Penny Wong')
    ORDER BY name, id DESC
""").df()

print("Panelist entries (newest first by ID):")
print(validation_result)

# Check for duplicates
duplicates = con.execute("""
    SELECT name, COUNT(*) as count 
    FROM dim_panellist 
    WHERE name IS NOT NULL 
    GROUP BY name 
    HAVING COUNT(*) > 1 
    ORDER BY count DESC 
    LIMIT 10
""").df()

print(f"\nFound {len(duplicates)} panelists with multiple entries:")
print(duplicates)

In [ ]:
# Check using actual episode dates instead of IDs
date_validation = con.execute("""
    WITH panelist_episodes AS (
        SELECT DISTINCT
            fr.speaker_name,
            dp.id as panelist_id,
            dp.profession,
            dp.link,
            de.date as episode_date
        FROM fact_responses fr
        JOIN dim_panellist dp ON fr.speaker_name = dp.name
        JOIN dim_episode de ON fr.episode_id = de.id
        WHERE fr.speaker_type = 3 
        AND fr.speaker_name IN ('Christopher Pyne', 'Malcolm Turnbull', 'Penny Wong')
    ),
    latest_by_date AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY speaker_name ORDER BY episode_date DESC, panelist_id DESC) as rn
        FROM panelist_episodes
    )
    SELECT speaker_name, panelist_id, profession, episode_date, rn
    FROM latest_by_date
    WHERE speaker_name IN ('Christopher Pyne', 'Malcolm Turnbull', 'Penny Wong')
    ORDER BY speaker_name, episode_date DESC, panelist_id DESC
""").df()

print("Panelist entries ordered by actual episode dates:")
print(date_validation)

# Check which entry is "latest" by date vs by ID
latest_by_date = date_validation[date_validation['rn'] == 1]
print("\nLatest entries by episode date:")
print(latest_by_date[['speaker_name', 'panelist_id', 'profession', 'episode_date']])

In [10]:
df_test = con.execute("""
    SELECT * FROM dim_panellist LIMIT 10
""").df()

In [ ]:
df_test.head().T


In [ ]:
# Check if speaker names match between tables
name_check = con.execute("""
    SELECT DISTINCT fr.speaker_name as response_name,
           dp.name as panelist_name
    FROM fact_responses fr
    FULL OUTER JOIN dim_panellist dp ON fr.speaker_name = dp.name
    WHERE fr.speaker_name IN ('Christopher Pyne', 'Malcolm Turnbull', 'Penny Wong')
       OR dp.name IN ('Christopher Pyne', 'Malcolm Turnbull', 'Penny Wong')
    ORDER BY response_name, panelist_name
""").df()

print("Name matching check:")
print(name_check)

In [ ]:
# See what speaker names exist in fact_responses
actual_names = con.execute("""
    SELECT DISTINCT speaker_name, COUNT(*) as count
    FROM fact_responses 
    WHERE speaker_type = 3
    AND (UPPER(speaker_name) LIKE '%PYNE%' 
         OR UPPER(speaker_name) LIKE '%TURNBULL%' 
         OR UPPER(speaker_name) LIKE '%WONG%')
    GROUP BY speaker_name
    ORDER BY count DESC
""").df()

print("Actual speaker names in fact_responses:")
print(actual_names)

In [ ]:
# Try case-insensitive match
case_insensitive_check = con.execute("""
    SELECT DISTINCT 
        fr.speaker_name as response_name,
        dp.name as panelist_name,
        UPPER(fr.speaker_name) as response_upper,
        UPPER(dp.name) as panelist_upper
    FROM fact_responses fr
    FULL OUTER JOIN dim_panellist dp ON UPPER(fr.speaker_name) = UPPER(dp.name)
    WHERE UPPER(fr.speaker_name) LIKE '%PYNE%' 
       OR UPPER(fr.speaker_name) LIKE '%TURNBULL%' 
       OR UPPER(fr.speaker_name) LIKE '%WONG%'
       OR UPPER(dp.name) LIKE '%PYNE%'
       OR UPPER(dp.name) LIKE '%TURNBULL%'
       OR UPPER(dp.name) LIKE '%WONG%'
    ORDER BY response_name, panelist_name
""").df()

print("\nCase-insensitive matching:")
print(case_insensitive_check)

In [ ]:
# Check why panelists don't have matches in dim_panellist
non_matches = con.execute("""
    SELECT DISTINCT 
        fr.speaker_name,
        COUNT(*) as response_count
    FROM fact_responses fr
    LEFT JOIN dim_panellist dp ON UPPER(fr.speaker_name) = UPPER(dp.name)
    WHERE fr.speaker_type = 3 
    AND dp.name IS NULL  -- No match found
    GROUP BY fr.speaker_name
    ORDER BY response_count DESC
    LIMIT 20
""").df()

print("Panelists in fact_responses with NO match in dim_panellist:")
print(non_matches)

In [ ]:
# Compare matches vs non-matches for panelists
match_stats = con.execute("""
    SELECT 
        CASE WHEN dp.name IS NOT NULL THEN 'MATCHED' ELSE 'NO_MATCH' END as match_status,
        COUNT(DISTINCT fr.speaker_name) as unique_speakers,
        COUNT(*) as total_responses
    FROM fact_responses fr
    LEFT JOIN dim_panellist dp ON UPPER(fr.speaker_name) = UPPER(dp.name)
    WHERE fr.speaker_type = 3
    GROUP BY match_status
""").df()

print("\nMatch statistics for panelists:")
print(match_stats)

In [ ]:
# Check for potential typos like "CHRISOPHER PYNE"
typo_check = con.execute("""
    SELECT DISTINCT fr.speaker_name
    FROM fact_responses fr
    WHERE fr.speaker_type = 3
    AND (fr.speaker_name LIKE '%PYNE%' OR fr.speaker_name LIKE '%TURNBULL%')
    ORDER BY fr.speaker_name
""").df()

print("\nAll PYNE/TURNBULL variants in fact_responses:")
print(typo_check)

In [ ]:
# Get unmatched speakers with their frequency
unmatched_speakers = con.execute("""
    SELECT DISTINCT 
        fr.speaker_name as unmatched_name,
        COUNT(*) as response_count
    FROM fact_responses fr
    LEFT JOIN dim_panellist dp ON UPPER(fr.speaker_name) = UPPER(dp.name)
    WHERE fr.speaker_type = 3 
    AND dp.name IS NULL
    GROUP BY fr.speaker_name
    ORDER BY response_count DESC
""").df()

# Get all available panelist names
available_names = con.execute("""
    SELECT DISTINCT name as available_name
    FROM dim_panellist
    ORDER BY name
""").df()

print(f"Found {len(unmatched_speakers)} unmatched speakers")
print(f"Found {len(available_names)} available panelist names")

In [ ]:
from difflib import SequenceMatcher
import pandas as pd

def similarity(a, b):
    """Calculate similarity ratio between two strings"""
    return SequenceMatcher(None, a.upper(), b.upper()).ratio()

# Find likely matches for top unmatched speakers
likely_matches = []

for _, unmatched_row in unmatched_speakers.head(20).iterrows():
    unmatched = unmatched_row['unmatched_name']
    count = unmatched_row['response_count']
    
    # Calculate similarity with all available names
    matches = []
    for _, available_row in available_names.iterrows():
        available = available_row['available_name']
        sim_score = similarity(unmatched, available)
        
        if sim_score > 0.6:  # Only show decent matches
            matches.append({
                'unmatched': unmatched,
                'response_count': count,
                'potential_match': available,
                'similarity': round(sim_score, 3)
            })
    
    # Sort by similarity and keep top 3
    matches.sort(key=lambda x: x['similarity'], reverse=True)
    likely_matches.extend(matches[:3])

# Convert to DataFrame and display
matches_df = pd.DataFrame(likely_matches)
print("\nLikely matches (similarity > 0.6):")
print(matches_df.head(30))

In [ ]:
# Also check for substring matches (different approach)
substring_matches = []

for _, unmatched_row in unmatched_speakers.head(20).iterrows():
    unmatched = unmatched_row['unmatched_name']
    count = unmatched_row['response_count']
    
    # Split into words for partial matching
    unmatched_words = set(unmatched.upper().split())
    
    for _, available_row in available_names.iterrows():
        available = available_row['available_name']
        available_words = set(available.upper().split())
        
        # Check for common words
        common_words = unmatched_words.intersection(available_words)
        if len(common_words) > 0:
            match_ratio = len(common_words) / max(len(unmatched_words), len(available_words))
            
            if match_ratio > 0.5:  # At least 50% word overlap
                substring_matches.append({
                    'unmatched': unmatched,
                    'response_count': count,
                    'potential_match': available,
                    'common_words': ', '.join(common_words),
                    'word_match_ratio': round(match_ratio, 3)
                })

substring_df = pd.DataFrame(substring_matches)
print("\nSubstring/word matches:")
print(substring_df.head(20))

In [ ]:
# Enhanced substring matching with better logic
enhanced_matches = []

for _, unmatched_row in unmatched_speakers.head(30).iterrows():
    unmatched = unmatched_row['unmatched_name']
    count = unmatched_row['response_count']
    
    # Split into words and clean
    unmatched_words = [word.strip() for word in unmatched.upper().split()]
    
    for _, available_row in available_names.iterrows():
        available = available_row['available_name']
        available_words = [word.strip() for word in available.upper().split()]
        
        # Find common words
        common_words = set(unmatched_words).intersection(set(available_words))
        
        if len(common_words) > 0:
            # Different scoring for last names vs first names
            has_lastname_match = any(len(word) > 3 for word in common_words)  # Assume last names are longer
            word_coverage = len(common_words) / len(unmatched_words)
            
            # Prioritize matches with last names and good coverage
            if has_lastname_match and word_coverage >= 0.5:
                enhanced_matches.append({
                    'unmatched': unmatched,
                    'response_count': count,
                    'potential_match': available,
                    'common_words': ', '.join(sorted(common_words)),
                    'word_coverage': round(word_coverage, 3),
                    'confidence': 'HIGH' if word_coverage >= 0.7 else 'MEDIUM'
                })

# Sort by response count and confidence
enhanced_df = pd.DataFrame(enhanced_matches)
enhanced_df = enhanced_df.sort_values(['response_count', 'word_coverage'], ascending=[False, False])

print("Enhanced substring matches (focusing on high-confidence):")
print(enhanced_df.head(20))

In [ ]:
# Look for obvious patterns
pattern_matches = []

for _, unmatched_row in unmatched_speakers.head(30).iterrows():
    unmatched = unmatched_row['unmatched_name']
    count = unmatched_row['response_count']
    
    # Check for exact substring containment
    for _, available_row in available_names.iterrows():
        available = available_row['available_name']
        
        # Check if unmatched is contained in available or vice versa
        unmatched_clean = unmatched.upper().replace('.', '').replace(',', '')
        available_clean = available.upper().replace('.', '').replace(',', '')
        
        if (unmatched_clean in available_clean or 
            available_clean in unmatched_clean or
            # Check reversed order (First Last vs Last First)
            ' '.join(reversed(unmatched_clean.split())) == available_clean):
            
            pattern_matches.append({
                'unmatched': unmatched,
                'response_count': count,
                'potential_match': available,
                'match_type': 'SUBSTRING/CONTAINMENT'
            })

pattern_df = pd.DataFrame(pattern_matches)
print("\nExact pattern matches:")
print(pattern_df.head(15))